In [ ]:
# !wget -q https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt -O input.txt

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'

/home/arunkant/miniconda3/envs/nanogpt/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
with open('input.txt', 'r') as f:
    text = f.read()

In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [4]:
stoi = {c:i for i,c in enumerate(chars)}
itos = {i:c for i,c in enumerate(chars)}

stoi, itos

({'\n': 0,
  ' ': 1,
  '!': 2,
  '$': 3,
  '&': 4,
  "'": 5,
  ',': 6,
  '-': 7,
  '.': 8,
  '3': 9,
  ':': 10,
  ';': 11,
  '?': 12,
  'A': 13,
  'B': 14,
  'C': 15,
  'D': 16,
  'E': 17,
  'F': 18,
  'G': 19,
  'H': 20,
  'I': 21,
  'J': 22,
  'K': 23,
  'L': 24,
  'M': 25,
  'N': 26,
  'O': 27,
  'P': 28,
  'Q': 29,
  'R': 30,
  'S': 31,
  'T': 32,
  'U': 33,
  'V': 34,
  'W': 35,
  'X': 36,
  'Y': 37,
  'Z': 38,
  'a': 39,
  'b': 40,
  'c': 41,
  'd': 42,
  'e': 43,
  'f': 44,
  'g': 45,
  'h': 46,
  'i': 47,
  'j': 48,
  'k': 49,
  'l': 50,
  'm': 51,
  'n': 52,
  'o': 53,
  'p': 54,
  'q': 55,
  'r': 56,
  's': 57,
  't': 58,
  'u': 59,
  'v': 60,
  'w': 61,
  'x': 62,
  'y': 63,
  'z': 64},
 {0: '\n',
  1: ' ',
  2: '!',
  3: '$',
  4: '&',
  5: "'",
  6: ',',
  7: '-',
  8: '.',
  9: '3',
  10: ':',
  11: ';',
  12: '?',
  13: 'A',
  14: 'B',
  15: 'C',
  16: 'D',
  17: 'E',
  18: 'F',
  19: 'G',
  20: 'H',
  21: 'I',
  22: 'J',
  23: 'K',
  24: 'L',
  25: 'M',
  26: 'N',
  27:

In [5]:
def encode(s):
    return [stoi[c] for c in s]
def decode(s):
    return ''.join([itos[i] for i in s])

decode(encode("hello world"))

'hello world'

In [6]:
block_size = 256 # context length
embedding_dim = 384
head_size = 6
num_heads = 64

In [7]:
data = torch.tensor(encode(text), dtype=torch.long)
ninghty_percent = int(0.9*len(data))
train_data, val_data = data[:ninghty_percent], data[ninghty_percent:]


train_data[:10], val_data[:10]

(tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47]),
 tensor([12,  0,  0, 19, 30, 17, 25, 21, 27, 10]))

In [8]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    indeces = torch.randint(0, len(data)-block_size, (4, ))
    X = torch.stack([data[i:i+block_size] for i in indeces])
    Y = torch.stack([data[i+1:i+block_size+1] for i in indeces])
    return X, Y

X, Y = get_batch('train')
X, Y

(tensor([[44, 53, 56,  ..., 42,  1, 40],
         [58, 47, 50,  ..., 50, 53, 61],
         [51, 47, 57,  ...,  1, 46, 43],
         [61, 53, 59,  ..., 60, 43, 52]]),
 tensor([[53, 56,  1,  ...,  1, 40, 63],
         [47, 50,  1,  ..., 53, 61,  1],
         [47, 57, 57,  ..., 46, 43,  1],
         [53, 59, 50,  ..., 43, 52, 57]]))

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            target = targets.view(B*T)
            loss = F.cross_entropy(logits, target)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is B, T
        for _ in range(max_new_tokens):
            # get preds
            logits, loss = self(idx) # B, T, C
            logits = logits[:, -1, :]   # B, C
            probs = F.softmax(logits, dim=1)
            idx_next = torch.multinomial(probs, num_samples=1) # B, 1
            idx = torch.cat((idx, idx_next), dim=1) # B, T+1
        return idx


In [ ]:
m = BigramLanguageModel(vocab_size)

In [ ]:
optim = torch.optim.AdamW(params=m.parameters(), lr=1e-3)

In [ ]:
# # Optimize the model
# for steps in range(5000):
#     X, Y = get_batch('train')
#     logits, loss = m(X, Y)
#     optim.zero_grad(set_to_none=True)
#     loss.backward()
#     optim.step()

# print(loss)

In [9]:
T = 4
wei = torch.tril(torch.ones(T, T))
wei

tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])

In [ ]:
wei = wei / wei.sum(1, keepdim=True)
wei

In [ ]:
wei.sum(1, keepdim=True)

In [ ]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros(T, T)
wei = wei.masked_fill(tril == 0, float('-inf'))
wei

In [ ]:
F.softmax(wei, dim=1)

In [ ]:
class Head(nn.Module):
    def __init__(self, embedding_dim, head_size, block_size):
        super().__init__()
        self.key = nn.Linear(embedding_dim, head_size, bias=False)
        self.query = nn.Linear(embedding_dim, head_size, bias=False)
        self.value = nn.Linear(embedding_dim, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        v = self.value(x) # (B, T, head_size)
        kt = k.transpose(1, 2) # dim1 = T, dim2 = head_size
        wei = q@kt # (B, T, T)
        
        wei = wei.masked_fill(self.tril == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)

        return wei @ v # (B, T, head_size)
    



In [46]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim, head_size, num_heads, block_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(embedding_dim, head_size, block_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, embedding_dim)

    def forward(self, x):
        outputs = torch.cat([head(x) for head in self.heads], dim=-1) # B, T, num_heads * head_size
        return self.proj(outputs)


In [47]:
class Block(nn.Module):
    def __init__(self, embedding_dim, head_size, num_heads, block_size):
        super().__init__()
        self.mha = MultiHeadAttention(embedding_dim, head_size, num_heads, block_size)
        self.layer1 = nn.Linear(embedding_dim, 4 * embedding_dim)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(4 * embedding_dim, embedding_dim)
        self.norm1 = nn.LayerNorm(embedding_dim)
        self.norm2 = nn.LayerNorm(embedding_dim)

    def feed_forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x
    
    def forward(self, x):
        x = x + self.mha(self.norm1(x)) # skip connections
        x = x + self.feed_forward(self.norm2(x))
        return x

In [48]:
class ToyGPT(nn.Module):
    def __init__(self, block_size, vocab_size, embedding_dim, head_size, num_heads):
        super().__init__()
        self.embedding_table = nn.Embedding(vocab_size, embedding_dim)
        self.pos_embedding_table = nn.Embedding(block_size, embedding_dim)
        self.blocks = nn.Sequential(*[Block(embedding_dim, head_size, num_heads, block_size) for _ in range(4)])
        self.lm_head = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x, targets=None):
        B, T = x.shape
        tok_emb = self.embedding_table(x)
        pos = torch.arange(T, device=x.device)
        pos_embedding = self.pos_embedding_table(pos)
        x = tok_emb + pos_embedding
        x = self.blocks(x)
        logits = self.lm_head(x) # B, T, vocab_size
        B, T, C = logits.shape

        if targets is None:
            loss = None
        else:
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens, block_size):
        device = next(self.parameters()).device
        idx = idx.to(device)
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:] # B, T
            logits, _ = self(idx_cond) # B, T, C
            logits = logits[:, -1, :] # B, C
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1) # B, 1
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


In [49]:

m = ToyGPT(block_size, vocab_size, embedding_dim, head_size, num_heads)
m = m.to(device)

In [50]:
context = torch.tensor(encode(' '), dtype=torch.long, device=device).unsqueeze(0)
print(decode([x.item() for x in m.generate(context, 128, block_size)[0]]))

RuntimeError: Expected size for first two dimensions of batch2 tensor to be: [1, 256] but got: [1, 1].

In [21]:
batch_size = 32
max_iters = 1000

print(f"Training on: {device}")

for iter in range(max_iters):
    # 1. Grab a batch of data
    xb, yb = get_batch('train') # xb and yb have shape (batch_size, block_size)
    xb = xb.to(device)
    yb = yb.to(device)
    
    # 2. Evaluate the loss
    logits, loss = m(xb, yb)
    
    # 3. Clear the old gradients
    optimizer.zero_grad(set_to_none=True)
    
    # 4. Calculate the new gradients (backpropagation)
    loss.backward()
    
    # 5. Update the weights
    optimizer.step()
    
    if iter % 100 == 0:
        print(f"Step {iter}: Loss {loss.item():.4f}")

Training on: cuda


RuntimeError: Expected all tensors to be on the same device, but got mat2 is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_mm)

In [22]:
for name, param in m.named_parameters():
    if param.device.type != 'cuda': # or 'mps' if on Mac
        print(f"Stuck on CPU: {name}")